In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
netSeq = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))

In [3]:
X = torch.rand(2,20)
netSeq(X)

tensor([[ 0.1193,  0.0945, -0.1408, -0.0293,  0.0787,  0.2264,  0.2487, -0.0364,
          0.0667,  0.1645],
        [ 0.1386,  0.1678, -0.0658,  0.0621,  0.1276,  0.2655,  0.2530, -0.0257,
          0.0147,  0.1348]], grad_fn=<AddmmBackward0>)

任何层或者神经网络都应该是nn.Moudle的子类

`class`中 最重要的两个函数：`__init__` 和 `forward`  
`__init__` 一般用来定义层和参数

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)
    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))

In [5]:
netMod = MLP()
netMod(X)

tensor([[ 0.0927, -0.3497, -0.2602, -0.2683, -0.0640,  0.1464,  0.0736, -0.1178,
         -0.0260,  0.1995],
        [ 0.0861, -0.3399,  0.0322, -0.1985, -0.0697,  0.0044,  0.0713, -0.1784,
          0.0876,  0.1353]], grad_fn=<AddmmBackward0>)

### 参数管理

参数访问：

In [7]:
print(netSeq[2].state_dict())

OrderedDict({'weight': tensor([[ 0.0371, -0.0572,  0.0081,  ...,  0.0043, -0.0112, -0.0407],
        [-0.0190, -0.0297, -0.0527,  ...,  0.0290,  0.0162,  0.0347],
        [-0.0179, -0.0041,  0.0164,  ...,  0.0044, -0.0158,  0.0283],
        ...,
        [-0.0181, -0.0605,  0.0291,  ..., -0.0477, -0.0233,  0.0096],
        [ 0.0471,  0.0327, -0.0010,  ..., -0.0316, -0.0095,  0.0377],
        [-0.0025, -0.0587, -0.0189,  ...,  0.0459,  0.0099,  0.0049]]), 'bias': tensor([ 0.0370,  0.0071,  0.0105,  0.0325,  0.0417,  0.0335,  0.0542, -0.0408,
        -0.0029,  0.0620])})


In [9]:
# torch.nn.parameter.Parameter 表示可优化的参数
print(type(netSeq[2].bias))

print(netSeq[2].bias)
# 可以通过data 真正访问该值
print(netSeq[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([ 0.0370,  0.0071,  0.0105,  0.0325,  0.0417,  0.0335,  0.0542, -0.0408,
        -0.0029,  0.0620], requires_grad=True)
tensor([ 0.0370,  0.0071,  0.0105,  0.0325,  0.0417,  0.0335,  0.0542, -0.0408,
        -0.0029,  0.0620])


In [13]:
# 可以通过data 访问梯度
# 由于还没有进行梯度计算，所以是None
netSeq[2].weight.grad == None

True

In [20]:
print(*[(name,param.shape) for name, param in netSeq[0].named_parameters()])
print(*[(name,param.shape) for name, param in netSeq.named_parameters()])

('weight', torch.Size([256, 20])) ('bias', torch.Size([256]))
('0.weight', torch.Size([256, 20])) ('0.bias', torch.Size([256])) ('2.weight', torch.Size([10, 256])) ('2.bias', torch.Size([10]))


可以通过访问索引字典键.data属性来访问

In [21]:
netSeq.state_dict()['2.bias'].data

tensor([ 0.0370,  0.0071,  0.0105,  0.0325,  0.0417,  0.0335,  0.0542, -0.0408,
        -0.0029,  0.0620])

nn.Sequence() 可以调用add_module 方法添加层

In [30]:

def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,4),nn.ReLU())
def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return net
rgnet = nn.Sequential(block2(),nn.Linear(4,1))
X = torch.rand(4)
rgnet(X)

tensor([-0.4349], grad_fn=<ViewBackward0>)

In [31]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


#### 参数初始化

**normal\_和zeros\_** 下划线代表原地操作，直接修改传入的数据

In [36]:
def init_weight(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)
rgnet.apply(init_weight)
rgnet[1].weight.data,rgnet[1].bias.data

(tensor([[-0.0107,  0.0129, -0.0014,  0.0130]]), tensor([0.]))

In [ ]:
nn.init.xavier_uniform_(m.weight)

#### 自定义层

自定义层和自定义网络没有本质区别，都是使用nn.Module抽象类

定义参数使用 nn.Parameter 类

In [44]:
## 自定义线性层
class My_Linear(nn.Module):
    def __init__(self,inunit,outunit):
        super().__init__()
        # randn 生成的是标志正态分布，rand 生成均匀分布
        self.weight = nn.Parameter(torch.randn(inunit,outunit))
        self.bias = nn.Parameter(torch.randn(outunit))
    def forward(self,X):
        linear = torch.matmul(X,self.weight.data)+self.bias.data
        return F.relu(linear)
dense = My_Linear(5,3)
dense.weight,dense.weight.data,dense.bias

(Parameter containing:
 tensor([[ 0.4145,  0.5343, -0.8266],
         [ 0.4529, -1.1252,  0.1741],
         [-0.2045,  0.7229, -0.7100],
         [-0.7691,  0.0532,  1.5087],
         [-0.3842,  0.9058, -0.1254]], requires_grad=True),
 tensor([[ 0.4145,  0.5343, -0.8266],
         [ 0.4529, -1.1252,  0.1741],
         [-0.2045,  0.7229, -0.7100],
         [-0.7691,  0.0532,  1.5087],
         [-0.3842,  0.9058, -0.1254]]),
 Parameter containing:
 tensor([ 1.0907,  0.2263, -0.1206], requires_grad=True))

#### torch文件读取

In [46]:
x = torch.ones(4)
y = torch.zeros(4)
torch.save([x,y],'torch_test_files')
x,y = torch.load('torch_test_files')
x,y

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38948\32342152.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x,y = torch.load('torch_test_files')


(tensor([1., 1., 1., 1.]), tensor([0., 0., 0., 0.]))

In [47]:
test_sv_dict = {'x':x,'y':y}
torch.save(test_sv_dict,'torch_dic_test_files')
test_sv_dict = torch.load('torch_dic_test_files')
test_sv_dict

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38948\688293147.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_sv_dict = torch.load('torch_dic_test_files')


{'x': tensor([1., 1., 1., 1.]), 'y': tensor([0., 0., 0., 0.])}

In [48]:
test_sv_dict = {1:x,2:y}
torch.save(test_sv_dict,'torch_dic_test_files')
test_sv_dict = torch.load('torch_dic_test_files')
test_sv_dict

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38948\1179105435.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_sv_dict = torch.load('torch_dic_test_files')


{1: tensor([1., 1., 1., 1.]), 2: tensor([0., 0., 0., 0.])}

In [51]:
class MLP(nn.Module):
    def __init__(self,in_units,hidden_units,out_units):
        super().__init__()
        self.hidden = nn.Linear(in_units,hidden_units)
        self.output = nn.Linear(hidden_units,out_units)
    def forward(self,X):
        return self.output(F.relu(self.hidden(X)))   

In [52]:
net = MLP(20,256,10)
x = torch.randn(2,20)
net(x)

tensor([[ 0.4612,  0.1969, -0.1729, -0.2450, -0.0840, -0.2663,  0.1965, -0.2219,
          0.1145, -0.1770],
        [ 0.4323, -0.1624, -0.4236,  0.3187, -0.0606, -0.4729, -0.0755,  0.2278,
          0.2145, -0.1766]], grad_fn=<AddmmBackward0>)

In [62]:
net.state_dict()

OrderedDict([('hidden.weight',
              tensor([[ 0.0404,  0.1861,  0.1152,  ..., -0.1884,  0.1941,  0.1945],
                      [-0.0606,  0.0878, -0.0204,  ..., -0.0343, -0.1275, -0.1932],
                      [-0.1504, -0.2113, -0.1205,  ..., -0.1994, -0.1872,  0.0425],
                      ...,
                      [ 0.0187, -0.1782,  0.1199,  ..., -0.1742, -0.0687, -0.1398],
                      [-0.1848,  0.0141,  0.1711,  ...,  0.1916, -0.0058, -0.0422],
                      [ 0.2167, -0.0918,  0.0463,  ...,  0.0010,  0.0813,  0.1496]])),
             ('hidden.bias',
              tensor([-0.0618, -0.1356, -0.1080,  0.1161, -0.0395,  0.1023,  0.1693,  0.0397,
                       0.0564, -0.1368, -0.0596, -0.0245, -0.1285,  0.2059,  0.1574,  0.0798,
                       0.0223,  0.0032,  0.2093,  0.2189, -0.0427,  0.0283,  0.0876, -0.2227,
                       0.2056,  0.1154,  0.0646, -0.0189, -0.0156,  0.1034, -0.1341,  0.2148,
                       0.1745,

In [53]:
torch.save(net.state_dict(),"mlp_params")

In [54]:
net_load = MLP(20,256,10)
net_load.load_state_dict(torch.load("mlp_params"))
net_load.eval()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38948\3242230748.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net_load.load_state_dict(torch.load("mlp_params"))


MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [55]:
net_load(x)==net(x)

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])